# Phân tích thống kê keypoints VSL400

Gắn hai dataset vào Input:

- `vsl400-train-keypoints`
- `vsl400-test-keypoints`

Notebook tìm mọi thư mục `train/` và `test/` dưới `/kaggle/input`. Cùng `split/nhãn/tên_file.npy` thì giữ file lớn hơn.

Mỗi file `.npy` là một clip, shape `(T, 76, 3)`. 34 điểm thân, sau đó 21 điểm tay trái và 21 điểm tay phải xếp xen kẽ theo tên khớp (`wrist_0`, `wrist_1`, ...). `_0` là tay trái, `_1` là tay phải. `x`, `y` là toạ độ MediaPipe trên khung hình (`y` tăng xuống dưới), `z` là độ sâu tương đối. Khớp không bắt được là đúng `(0, 0, 0)`. File `(T, 228)` được reshape về `(T, 76, 3)`.

Bốn phần:

1. Số clip từng nhãn, train và test.
2. Số frame từng clip, đối chiếu mốc 80 frame mà model đang giữ.
3. Tỷ lệ điểm `(0, 0, 0)` theo khớp, theo thân/tay, theo nhãn.
4. Mean, std và phân vị của `x`, `y`, `z` trên điểm khác `(0, 0, 0)`, để lấy bộ chuẩn hoá chung. Bộ số fit trên **train**.

Không cần Internet, không cần GPU.


In [ ]:
import json
import logging
import math
import os
import time
import unicodedata
from multiprocessing import get_context
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    force=True,
)
logger = logging.getLogger('keypoint_stats')

INPUT_DIR = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/tmp/vsl400-keypoint-stats')
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ('train', 'test')
SKIP_DIR_NAMES = {'mediapipe_models'}
N_JOINTS = 76
MAX_FRAMES = 80
LABEL_PAGE_SIZE = 40
EXTREME_GLOSS_COUNT = 30

# Histogram covers MediaPipe x, y and relative z. Width is 0.01.
EDGES = np.linspace(-4.0, 4.0, 801)
N_BINS = int(EDGES.size - 1)

BODY_LANDMARKS = [
    'nose', 'leftEyeInner', 'leftEye', 'leftEyeOuter', 'rightEyeInner', 'rightEye', 'rightEyeOuter',
    'leftEar', 'rightEar', 'mouthLeft', 'mouthRight', 'leftShoulder', 'rightShoulder',
    'leftElbow', 'rightElbow', 'leftWrist', 'rightWrist', 'leftPinky', 'rightPinky',
    'leftIndex', 'rightIndex', 'leftThumb', 'rightThumb',
    'leftHip', 'rightHip', 'leftKnee', 'rightKnee', 'leftAnkle', 'rightAnkle',
    'leftHeel', 'rightHeel', 'leftFootIndex', 'rightFootIndex', 'neck',
]
HAND_LANDMARKS = [
    'wrist', 'indexTip', 'indexDIP', 'indexPIP', 'indexMCP',
    'middleTip', 'middleDIP', 'middlePIP', 'middleMCP',
    'ringTip', 'ringDIP', 'ringPIP', 'ringMCP',
    'littleTip', 'littleDIP', 'littlePIP', 'littleMCP',
    'thumbTip', 'thumbIP', 'thumbMP', 'thumbCMC',
]
LANDMARKS = BODY_LANDMARKS + [name + suffix for name in HAND_LANDMARKS for suffix in ('_0', '_1')]
if len(LANDMARKS) != N_JOINTS:
    raise RuntimeError(f'expected {N_JOINTS} landmarks, got {len(LANDMARKS)}')

def pretty_landmark(name: str) -> str:
    if name.endswith('_0'):
        return name[:-2] + ' left'
    if name.endswith('_1'):
        return name[:-2] + ' right'
    return name

PRETTY_LANDMARKS = [pretty_landmark(name) for name in LANDMARKS]
BODY_INDEX = np.arange(0, 34, dtype=np.int64)
LEFT_INDEX = np.arange(34, 76, 2, dtype=np.int64)
RIGHT_INDEX = np.arange(35, 76, 2, dtype=np.int64)
GROUP_INDICES = (BODY_INDEX, LEFT_INDEX, RIGHT_INDEX)
GROUP_NAMES = ('body', 'left_hand', 'right_hand')
AXIS_NAMES = ('x', 'y', 'z')
if tuple(len(index) for index in GROUP_INDICES) != (34, 21, 21):
    raise RuntimeError('joint groups do not partition 76 landmarks')

TRAIN_COLOR = '#4C78A8'
TEST_COLOR = '#E45756'
BODY_COLOR = '#4C78A8'
LEFT_COLOR = '#F58518'
RIGHT_COLOR = '#54A24B'
GROUP_COLORS = (BODY_COLOR, LEFT_COLOR, RIGHT_COLOR)
LANDMARK_COLORS = (
    [BODY_COLOR] * len(BODY_INDEX)
    + [color for _name in HAND_LANDMARKS for color in (LEFT_COLOR, RIGHT_COLOR)]
)

BODY_EDGES = (
    (0, 2), (0, 5), (2, 7), (5, 8),
    (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
    (11, 23), (12, 24), (23, 24),
    (23, 25), (25, 27), (24, 26), (26, 28),
    (27, 29), (29, 31), (28, 30), (30, 32),
    (0, 33), (11, 33), (12, 33),
    (15, 17), (15, 19), (15, 21),
    (16, 18), (16, 20), (16, 22),
)
# Local hand index follows HAND_LANDMARKS, tip back to the wrist.
FINGER_CHAINS = (
    (1, 2, 3, 4, 0),
    (5, 6, 7, 8, 0),
    (9, 10, 11, 12, 0),
    (13, 14, 15, 16, 0),
    (17, 18, 19, 20, 0),
)

def hand_joint(local: int, side: int) -> int:
    return 34 + local * 2 + side

def canonical_name(name: str) -> str:
    return unicodedata.normalize('NFC', name)

def save_show(fig, name: str) -> None:
    import matplotlib.pyplot as plt
    path = FIGURE_DIR / name
    fig.savefig(path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    logger.info('Saved %s', path)

def style_plot() -> None:
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.grid': True,
        'grid.alpha': 0.25,
        'axes.unicode_minus': False,
        'font.size': 11,
    })

def reshape_keypoints(array: np.ndarray) -> np.ndarray:
    if array.ndim == 3 and tuple(array.shape[1:]) == (N_JOINTS, 3):
        return array
    if array.ndim == 2 and array.shape[1] == N_JOINTS * 3:
        return array.reshape(array.shape[0], N_JOINTS, 3)
    raise ValueError(f'expected (T, {N_JOINTS}, 3) or (T, {N_JOINTS * 3}), got {tuple(array.shape)}')

def _histogram(values: np.ndarray) -> tuple[np.ndarray, int, int]:
    if values.size == 0:
        return np.zeros(N_BINS, dtype=np.int64), 0, 0
    counts, _edges = np.histogram(values, bins=EDGES)
    under = int(np.count_nonzero(values < EDGES[0]))
    over = int(np.count_nonzero(values > EDGES[-1]))
    return counts.astype(np.int64, copy=False), under, over

def hist_percentile(counts: np.ndarray, edges: np.ndarray, underflow: int, overflow: int, quantile: float) -> float:
    total = int(counts.sum()) + int(underflow) + int(overflow)
    if total <= 0:
        return float('nan')
    target = float(quantile) * total
    if target <= underflow:
        return float(edges[0])
    running = float(underflow)
    for count, left, right in zip(counts, edges[:-1], edges[1:]):
        count = float(count)
        if running + count >= target:
            if count == 0:
                return float(left)
            frac = (target - running) / count
            return float(left + frac * (right - left))
        running += count
    return float(edges[-1])

def hist_fraction_below(counts: np.ndarray, edges: np.ndarray, underflow: int, overflow: int, threshold: float) -> float:
    total = float(counts.sum() + underflow + overflow)
    if total <= 0 or threshold <= float(edges[0]):
        return float('nan')
    below = float(underflow)
    for count, left, right in zip(counts, edges[:-1], edges[1:]):
        if right <= threshold:
            below += float(count)
        elif left < threshold:
            width = float(right - left)
            below += float(count) * ((threshold - left) / width)
        else:
            break
    return below / total

def hist_fraction_above(counts: np.ndarray, edges: np.ndarray, underflow: int, overflow: int, threshold: float) -> float:
    total = float(counts.sum() + underflow + overflow)
    if total <= 0 or threshold >= float(edges[-1]):
        return float('nan')
    above = float(overflow)
    for count, left, right in zip(counts[::-1], edges[:-1][::-1], edges[1:][::-1]):
        if left >= threshold:
            above += float(count)
        elif right > threshold:
            width = float(right - left)
            above += float(count) * ((right - threshold) / width)
        else:
            break
    return above / total

def density_from_hist(counts: np.ndarray, edges: np.ndarray) -> np.ndarray:
    widths = np.diff(edges)
    total = float(np.asarray(counts).sum())
    if total <= 0:
        return np.zeros(len(widths), dtype=np.float64)
    return np.asarray(counts, dtype=np.float64) / (total * widths)

def _analyze_file(split: str, gloss: str, path: str) -> dict:
    array = reshape_keypoints(np.load(path, allow_pickle=False))
    if array.shape[0] == 0:
        raise ValueError('empty array')
    if not np.isfinite(array).all():
        nonfinite = int(np.size(array) - np.isfinite(array).sum())
        array = np.where(np.isfinite(array), array, np.float32(0))
    else:
        nonfinite = 0
    if array.dtype != np.float32:
        array = array.astype(np.float32)
    zero = np.all(array == 0, axis=-1)
    valid = ~zero
    near = np.all(np.abs(array) <= np.float32(1e-6), axis=-1)
    near_extra = int(np.count_nonzero(near & valid))
    values = array.astype(np.float64)
    masked = np.where(valid[:, :, None], values, 0.0)
    coord_sum = masked.sum(axis=0)
    coord_sumsq = np.square(masked).sum(axis=0)
    coord_min = np.where(valid[:, :, None], values, np.inf).min(axis=0)
    coord_max = np.where(valid[:, :, None], values, -np.inf).max(axis=0)
    hist = np.zeros((len(GROUP_INDICES), 3, N_BINS), dtype=np.int64)
    underflow = np.zeros((len(GROUP_INDICES), 3), dtype=np.int64)
    overflow = np.zeros((len(GROUP_INDICES), 3), dtype=np.int64)
    for group_id, index in enumerate(GROUP_INDICES):
        group_valid = valid[:, index]
        group_values = values[:, index, :]
        for axis in range(3):
            selected = group_values[:, :, axis][group_valid]
            counts, under, over = _histogram(selected)
            hist[group_id, axis] = counts
            underflow[group_id, axis] = under
            overflow[group_id, axis] = over
    return {
        'ok': True,
        'split': split,
        'gloss': gloss,
        'path': path,
        'n_frames': int(array.shape[0]),
        'zero_counts': zero.sum(axis=0).astype(np.int64),
        'valid_counts': valid.sum(axis=0).astype(np.int64),
        'coord_sum': coord_sum,
        'coord_sumsq': coord_sumsq,
        'coord_min': coord_min,
        'coord_max': coord_max,
        'hist': hist,
        'underflow': underflow,
        'overflow': overflow,
        'body_present': int((~zero[:, BODY_INDEX].all(axis=1)).sum()),
        'left_present': int((~zero[:, LEFT_INDEX].all(axis=1)).sum()),
        'right_present': int((~zero[:, RIGHT_INDEX].all(axis=1)).sum()),
        'all_zero_frames': int(zero.all(axis=1).sum()),
        'near_extra': near_extra,
        'nonfinite': nonfinite,
    }

def analyze_file(job: tuple[str, str, str]) -> dict:
    split, gloss, path = job
    try:
        return _analyze_file(split, gloss, path)
    except Exception as exc:
        return {
            'ok': False,
            'split': split,
            'gloss': gloss,
            'path': path,
            'reason': f'{type(exc).__name__}: {exc}',
        }

class RunningStats:
    def __init__(self) -> None:
        self.files = 0
        self.frames = 0
        self.all_zero_frames = 0
        self.body_present = 0
        self.left_present = 0
        self.right_present = 0
        self.near_extra = 0
        self.nonfinite = 0
        self.zero_counts = np.zeros(N_JOINTS, dtype=np.int64)
        self.valid_counts = np.zeros(N_JOINTS, dtype=np.int64)
        self.coord_sum = np.zeros((N_JOINTS, 3), dtype=np.float64)
        self.coord_sumsq = np.zeros((N_JOINTS, 3), dtype=np.float64)
        self.coord_min = np.full((N_JOINTS, 3), np.inf, dtype=np.float64)
        self.coord_max = np.full((N_JOINTS, 3), -np.inf, dtype=np.float64)
        self.hist = np.zeros((len(GROUP_INDICES), 3, N_BINS), dtype=np.int64)
        self.underflow = np.zeros((len(GROUP_INDICES), 3), dtype=np.int64)
        self.overflow = np.zeros((len(GROUP_INDICES), 3), dtype=np.int64)

    def add(self, item: dict) -> None:
        self.files += 1
        self.frames += int(item['n_frames'])
        self.all_zero_frames += int(item['all_zero_frames'])
        self.body_present += int(item['body_present'])
        self.left_present += int(item['left_present'])
        self.right_present += int(item['right_present'])
        self.near_extra += int(item['near_extra'])
        self.nonfinite += int(item['nonfinite'])
        self.zero_counts += item['zero_counts']
        self.valid_counts += item['valid_counts']
        self.coord_sum += item['coord_sum']
        self.coord_sumsq += item['coord_sumsq']
        self.coord_min = np.minimum(self.coord_min, item['coord_min'])
        self.coord_max = np.maximum(self.coord_max, item['coord_max'])
        self.hist += item['hist']
        self.underflow += item['underflow']
        self.overflow += item['overflow']

def find_split_dirs(input_dir: Path) -> dict[str, list[Path]]:
    found = {split: [] for split in SPLITS}
    if not input_dir.is_dir():
        return found
    for dirpath, dirnames, _filenames in os.walk(input_dir):
        dirnames[:] = [
            name
            for name in dirnames
            if name not in SKIP_DIR_NAMES and not name.startswith('.')
        ]
        base = os.path.basename(dirpath)
        if base not in SPLITS:
            continue
        found[base].append(Path(dirpath))
        dirnames.clear()
    for split in SPLITS:
        found[split].sort(key=lambda path: str(path))
    return found

def collect_samples(split_dir: Path) -> list[tuple[str, str, str]]:
    rows = []
    split_name = split_dir.name
    loose = [path for path in split_dir.glob('*.npy') if path.is_file()]
    if loose:
        logger.warning('Skipping %d npy files sitting directly in %s', len(loose), split_dir)
    gloss_dirs = sorted(
        (path for path in split_dir.iterdir() if path.is_dir() and not path.name.startswith('.')),
        key=lambda path: path.name,
    )
    for gloss_dir in gloss_dirs:
        if gloss_dir.name in SKIP_DIR_NAMES:
            continue
        direct = [path for path in gloss_dir.glob('*.npy') if path.is_file()]
        if direct:
            gloss_name = canonical_name(gloss_dir.name)
            for path in direct:
                rows.append((split_name, gloss_name, str(path)))
            continue
        children = sorted(
            (path for path in gloss_dir.iterdir() if path.is_dir() and not path.name.startswith('.')),
            key=lambda path: path.name,
        )
        for child in children:
            if child.name in SKIP_DIR_NAMES:
                continue
            nested = [path for path in child.glob('*.npy') if path.is_file()]
            if not nested:
                continue
            gloss_name = canonical_name(child.name)
            for path in nested:
                rows.append((split_name, gloss_name, str(path)))
    return rows

def zero_rate(running: RunningStats, indices: np.ndarray) -> float:
    total = running.frames * int(len(indices))
    if total <= 0:
        return float('nan')
    return float(running.zero_counts[indices].sum() / total)

def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, (np.floating, float)):
        number = float(value)
        if not math.isfinite(number):
            return None
        return number
    if isinstance(value, (np.integer, int)) and not isinstance(value, bool):
        return int(value)
    return value

logger.info('Output directory: %s', OUTPUT_DIR)


## 1. Số clip mỗi nhãn

Đếm file theo thư mục nhãn. Bảng đầy đủ nằm ngay dưới, biểu đồ ở sau khi quét xong nội dung `.npy` (pool đọc file xong rồi mới vẽ, tránh fork sau matplotlib).


In [ ]:
if INPUT_DIR.is_dir():
    logger.info('Mounted inputs: %s', sorted(path.name for path in INPUT_DIR.iterdir()))
else:
    logger.error('Nothing is mounted at %s. Attach the train and test keypoint datasets.', INPUT_DIR)

split_dirs = find_split_dirs(INPUT_DIR)
for split in SPLITS:
    roots = split_dirs[split]
    if not roots:
        logger.warning('No %s directory under %s', split, INPUT_DIR)
    if len(roots) > 1:
        logger.warning('Found %d %s roots. Same gloss/filename keeps the larger file.', len(roots), split)
    for root in roots:
        logger.info('Split root %s: %s', split, root)

raw_rows = []
for split in SPLITS:
    for root in split_dirs[split]:
        raw_rows.extend(collect_samples(root))

best = {}
duplicate_count = 0
for split, gloss, path in raw_rows:
    key = (split, gloss, canonical_name(Path(path).name))
    size = os.path.getsize(path)
    previous = best.get(key)
    if previous is None:
        best[key] = (size, path, split, gloss)
        continue
    duplicate_count += 1
    if size > previous[0]:
        best[key] = (size, path, split, gloss)

jobs = [(split, gloss, path) for _size, path, split, gloss in best.values()]
jobs.sort()
if not jobs:
    raise RuntimeError(
        'No npy files under train/ or test/. Attach vsl400-train-keypoints and vsl400-test-keypoints.'
    )

catalog = pd.DataFrame(jobs, columns=['split', 'gloss', 'path'])
label_counts = (
    catalog.groupby(['gloss', 'split'])
    .size()
    .unstack(fill_value=0)
)
for split in SPLITS:
    if split not in label_counts.columns:
        label_counts[split] = 0
label_counts = label_counts[list(SPLITS)]
label_counts['total'] = label_counts['train'] + label_counts['test']
label_counts = label_counts.sort_index()
label_counts.index.name = 'gloss'

logger.info('Files after dedupe: %d (dropped %d duplicate names)', len(jobs), duplicate_count)
logger.info('Glosses: %d', int(label_counts.shape[0]))
for split in SPLITS:
    logger.info(
        '%s files=%d glosses=%d',
        split,
        int(label_counts[split].sum()),
        int((label_counts[split] > 0).sum()),
    )

train_positive = label_counts.loc[label_counts['train'] > 0, 'train']
if not train_positive.empty:
    logger.info(
        'Train clips per gloss: min=%d median=%.0f max=%d',
        int(train_positive.min()),
        float(train_positive.median()),
        int(train_positive.max()),
    )

missing_train = label_counts.index[label_counts['train'] == 0].tolist()
missing_test = label_counts.index[label_counts['test'] == 0].tolist()
logger.info('Glosses missing from train: %d', len(missing_train))
logger.info('Glosses missing from test: %d', len(missing_test))
if missing_train:
    logger.info('Missing train sample: %s', ', '.join(missing_train[:30]))
if missing_test:
    logger.info('Missing test sample: %s', ', '.join(missing_test[:30]))

label_path = OUTPUT_DIR / 'label_counts.csv'
label_counts.to_csv(label_path, encoding='utf-8')
logger.info('Wrote %s', label_path)
with pd.option_context('display.max_rows', 500, 'display.max_colwidth', 80):
    display(label_counts)


## 2. Quét từng file keypoints

Một process pool đọc mỗi `.npy` đúng một lần. Mean, std, min, max là số đúng trên mọi điểm khác `(0, 0, 0)`. Phân vị lấy từ histogram bước 0.01 trong đoạn `[-4, 4]`.


In [ ]:
preview = np.load(jobs[0][2], mmap_mode='r')
logger.info('Preview %s shape=%s dtype=%s', jobs[0][2], tuple(preview.shape), preview.dtype)
del preview

rows = []
bad = []
overall = RunningStats()
by_split = {split: RunningStats() for split in SPLITS}
worker_count = max(1, min(len(jobs), os.cpu_count() or 1))
chunksize = max(1, min(32, len(jobs) // (worker_count * 4) or 1))
logger.info('Scanning %d files with %d workers, chunksize %d', len(jobs), worker_count, chunksize)
started = time.perf_counter()
with get_context('fork').Pool(processes=worker_count) as pool:
    iterator = pool.imap_unordered(analyze_file, jobs, chunksize=chunksize)
    for item in tqdm(iterator, total=len(jobs), desc='keypoints'):
        if not item['ok']:
            bad.append(item)
            continue
        split = item['split']
        if split not in by_split:
            bad.append({'ok': False, 'path': item['path'], 'reason': f'unexpected split {split}'})
            continue
        overall.add(item)
        by_split[split].add(item)
        rows.append({
            'split': split,
            'gloss': item['gloss'],
            'path': item['path'],
            'n_frames': int(item['n_frames']),
            'all_zero_frames': int(item['all_zero_frames']),
            'zero_points': int(item['zero_counts'].sum()),
            'body_present_frames': int(item['body_present']),
            'left_present_frames': int(item['left_present']),
            'right_present_frames': int(item['right_present']),
        })

elapsed = time.perf_counter() - started
for item in bad[:20]:
    logger.error('Bad file %s: %s', item.get('path'), item.get('reason'))
if bad:
    logger.error('Unreadable files: %d', len(bad))
if not rows:
    detail = '; '.join(str(item.get('reason', '')) for item in bad[:8])
    raise RuntimeError(f'Every keypoint file failed. {detail}')

files_df = pd.DataFrame(rows)
files_df['content_frames'] = files_df['n_frames'] - files_df['all_zero_frames']
files_df['zero_ratio'] = files_df['zero_points'] / (files_df['n_frames'] * N_JOINTS)
files_df['body_presence'] = files_df['body_present_frames'] / files_df['n_frames']
files_df['left_presence'] = files_df['left_present_frames'] / files_df['n_frames']
files_df['right_presence'] = files_df['right_present_frames'] / files_df['n_frames']
files_df = files_df.sort_values(['split', 'gloss', 'path']).reset_index(drop=True)

point_total = overall.frames * N_JOINTS
logger.info(
    'Scanned %d files in %.1f s. Frames=%d all-zero-frames=%d zero-rate=%.4f near-extra=%d nonfinite=%d',
    len(files_df),
    elapsed,
    overall.frames,
    overall.all_zero_frames,
    float(overall.zero_counts.sum() / point_total),
    overall.near_extra,
    overall.nonfinite,
)
length_share = files_df['n_frames'].value_counts(normalize=True)
if float(length_share.iloc[0]) >= 0.8:
    logger.warning(
        '%.1f%% of clips have exactly %d frames. Length may already be trimmed or padded.',
        100 * float(length_share.iloc[0]),
        int(length_share.index[0]),
    )

stats_path = OUTPUT_DIR / 'file_stats.csv'
files_df.to_csv(stats_path, index=False, encoding='utf-8')
logger.info('Wrote %s', stats_path)


## Biểu đồ số lượng nhãn

Trang ngang là toàn bộ nhãn, sắp theo tên. Hai biểu đồ đầu và cuối chỉ rút nhãn nhiều clip nhất và ít clip nhất trên train.


In [ ]:
import matplotlib.pyplot as plt

style_plot()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sample_totals = [int(label_counts[split].sum()) for split in SPLITS]
gloss_totals = [int((label_counts[split] > 0).sum()) for split in SPLITS]
bars = axes[0].bar(list(SPLITS), sample_totals, color=[TRAIN_COLOR, TEST_COLOR])
axes[0].set_title('Tổng số clip')
axes[0].set_ylabel('Clip')
for bar, value in zip(bars, sample_totals):
    axes[0].annotate(str(value), xy=(bar.get_x() + bar.get_width() / 2, value), ha='center', va='bottom')
bars = axes[1].bar(list(SPLITS), gloss_totals, color=[TRAIN_COLOR, TEST_COLOR])
axes[1].set_title('Số nhãn có ít nhất 1 clip')
axes[1].set_ylabel('Nhãn')
for bar, value in zip(bars, gloss_totals):
    axes[1].annotate(str(value), xy=(bar.get_x() + bar.get_width() / 2, value), ha='center', va='bottom')
save_show(fig, 'labels_overview.png')

max_count = int(label_counts[list(SPLITS)].to_numpy().max())
bins = np.arange(0, max_count + 2) - 0.5
fig, ax = plt.subplots(figsize=(8, 4))
for split, color in (('train', TRAIN_COLOR), ('test', TEST_COLOR)):
    values = label_counts.loc[label_counts[split] > 0, split]
    if values.empty:
        continue
    ax.hist(values, bins=bins, alpha=0.55, label=split, color=color)
ax.set_xlabel('Số clip của một nhãn')
ax.set_ylabel('Số nhãn')
ax.set_title('Phân bố số clip trên mỗi nhãn')
ax.legend()
save_show(fig, 'labels_hist.png')

fig, ax = plt.subplots(figsize=(6, 6))
for split, color in (('train', TRAIN_COLOR), ('test', TEST_COLOR)):
    values = np.sort(label_counts[split].to_numpy(dtype=np.float64))
    if values.sum() <= 0:
        continue
    share = np.cumsum(values) / values.sum()
    rank = np.arange(1, values.size + 1) / values.size
    ax.plot(rank, share, color=color, label=split)
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', label='đều')
ax.set_xlabel('Tỷ lệ nhãn, xếp từ nhãn ít mẫu')
ax.set_ylabel('Tỷ lệ clip tích lũy')
ax.set_title('Độ lệch số mẫu giữa các nhãn')
ax.legend()
save_show(fig, 'labels_lorenz.png')

def plot_extreme(column: str, largest: bool, title: str, filename: str) -> None:
    if largest:
        chosen = label_counts[column].sort_values(ascending=False).head(EXTREME_GLOSS_COUNT)
    else:
        positive = label_counts.loc[label_counts[column] > 0, column]
        chosen = positive.sort_values(ascending=True).head(EXTREME_GLOSS_COUNT)
    if chosen.empty:
        logger.warning('Skip %s, no rows', filename)
        return
    glosses = list(chosen.index)
    y = np.arange(len(glosses))
    fig, ax = plt.subplots(figsize=(10, max(4, len(glosses) * 0.32)))
    ax.barh(y - 0.2, [label_counts.loc[name, 'train'] for name in glosses], 0.4, label='train', color=TRAIN_COLOR)
    ax.barh(y + 0.2, [label_counts.loc[name, 'test'] for name in glosses], 0.4, label='test', color=TEST_COLOR)
    ax.set_yticks(y)
    ax.set_yticklabels(glosses)
    ax.invert_yaxis()
    ax.set_xlabel('Số clip')
    ax.set_title(title)
    ax.legend()
    save_show(fig, filename)

if label_counts.shape[0] > LABEL_PAGE_SIZE:
    plot_extreme('train', True, f'{EXTREME_GLOSS_COUNT} nhãn nhiều clip train nhất', 'labels_top.png')
    plot_extreme('train', False, f'{EXTREME_GLOSS_COUNT} nhãn ít clip train nhất', 'labels_bottom.png')

glosses = list(label_counts.sort_index().index)
for page, start in enumerate(range(0, len(glosses), LABEL_PAGE_SIZE), start=1):
    chunk = glosses[start:start + LABEL_PAGE_SIZE]
    y = np.arange(len(chunk))
    fig, ax = plt.subplots(figsize=(10, max(4, len(chunk) * 0.28)))
    ax.barh(y - 0.2, [int(label_counts.loc[name, 'train']) for name in chunk], 0.4, label='train', color=TRAIN_COLOR)
    ax.barh(y + 0.2, [int(label_counts.loc[name, 'test']) for name in chunk], 0.4, label='test', color=TEST_COLOR)
    ax.set_yticks(y)
    ax.set_yticklabels(chunk)
    ax.invert_yaxis()
    ax.set_xlabel('Số clip')
    ax.set_title(f'Số clip mỗi nhãn ({start + 1}-{start + len(chunk)})')
    ax.legend()
    save_show(fig, f'labels_page_{page:02d}.png')


## 3. Số frame mỗi clip

Model đang giữ 80 frame đầu và pad phần thiếu bằng 0. Clip dài hơn 80 sẽ bị cắt, clip ngắn hơn sẽ thêm frame 0. Đường đứt trên histogram là mốc đó.


In [ ]:
def frame_table(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    parts = [('all', frame)] + [(split, frame[frame['split'] == split]) for split in SPLITS]
    for name, part in parts:
        if part.empty:
            continue
        values = part['n_frames'].to_numpy()
        row = {
            'split': name,
            'clips': int(values.size),
            'min': int(values.min()),
            'mean': float(values.mean()),
            'max': int(values.max()),
        }
        for quantile in (1, 5, 25, 50, 75, 95, 99):
            row[f'p{quantile:02d}'] = float(np.percentile(values, quantile))
        row['share_shorter_than_80'] = float(np.mean(values < MAX_FRAMES))
        row['share_longer_than_80'] = float(np.mean(values > MAX_FRAMES))
        row['frames_cut_if_keep_80'] = int(np.maximum(values - MAX_FRAMES, 0).sum())
        row['frames_padded_if_len_80'] = int(np.maximum(MAX_FRAMES - values, 0).sum())
        rows.append(row)
    return pd.DataFrame(rows)

frame_summary = frame_table(files_df)
display(frame_summary.round(3))
for row in frame_summary.to_dict(orient='records'):
    logger.info(
        '%s clips=%d median=%.0f mean=%.1f min=%d max=%d shorter80=%.1f%% longer80=%.1f%% cut_frames=%d pad_frames=%d',
        row['split'],
        row['clips'],
        row['p50'],
        row['mean'],
        row['min'],
        row['max'],
        100 * row['share_shorter_than_80'],
        100 * row['share_longer_than_80'],
        row['frames_cut_if_keep_80'],
        row['frames_padded_if_len_80'],
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split, color in (('train', TRAIN_COLOR), ('test', TEST_COLOR)):
    values = files_df.loc[files_df['split'] == split, 'n_frames'].to_numpy()
    if values.size == 0:
        continue
    axes[0].hist(values, bins=40, density=True, alpha=0.55, label=split, color=color)
    ordered = np.sort(values)
    ranks = np.arange(1, ordered.size + 1) / ordered.size
    axes[1].plot(ordered, ranks, color=color, label=split)
axes[0].axvline(MAX_FRAMES, color='black', linestyle='--', label='80')
axes[1].axvline(MAX_FRAMES, color='black', linestyle='--', label='80')
axes[0].set_xlabel('Số frame')
axes[0].set_ylabel('Mật độ')
axes[0].set_title('Phân bố độ dài clip')
axes[0].legend()
axes[1].set_xlabel('Số frame')
axes[1].set_ylabel('Tỷ lệ clip có độ dài <= x')
axes[1].set_title('ECDF độ dài clip')
axes[1].legend()
save_show(fig, 'frames_hist_ecdf.png')

fig, ax = plt.subplots(figsize=(6, 4))
box_data = []
box_labels = []
for split in SPLITS:
    values = files_df.loc[files_df['split'] == split, 'n_frames'].to_numpy()
    if values.size == 0:
        continue
    box_data.append(values)
    box_labels.append(split)
ax.boxplot(box_data, showfliers=False)
ax.set_xticks(range(1, len(box_labels) + 1))
ax.set_xticklabels(box_labels)
ax.axhline(MAX_FRAMES, color='black', linestyle='--', label='80')
ax.set_ylabel('Số frame')
ax.set_title('Độ dài clip, không vẽ outlier')
ax.legend()
save_show(fig, 'frames_box.png')

bin_edges = [0, 30, 60, 80, 120, 180, 10**9]
bin_labels = ['1-30', '31-60', '61-80', '81-120', '121-180', '181+']
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(bin_labels))
width = 0.38
for offset, split, color in ((-width / 2, 'train', TRAIN_COLOR), (width / 2, 'test', TEST_COLOR)):
    part = files_df.loc[files_df['split'] == split, 'n_frames']
    if part.empty:
        continue
    buckets = pd.cut(part, bins=bin_edges, labels=bin_labels, right=True)
    counts = buckets.value_counts().reindex(bin_labels, fill_value=0)
    ax.bar(x + offset, counts.to_numpy(), width=width, label=split, color=color)
ax.set_xticks(x)
ax.set_xticklabels(bin_labels)
ax.set_ylabel('Số clip')
ax.set_title('Clip theo khoảng độ dài')
ax.legend()
save_show(fig, 'frames_buckets.png')

def plot_gloss_length(largest: bool, filename: str, title: str) -> None:
    train_files = files_df[files_df['split'] == 'train']
    if train_files.empty:
        logger.warning('No train files for %s', filename)
        return
    grouped = train_files.groupby('gloss')['n_frames'].agg(['median', 'count'])
    grouped = grouped[grouped['count'] >= 3]
    if grouped.empty:
        grouped = train_files.groupby('gloss')['n_frames'].agg(['median', 'count'])
    chosen = grouped.nlargest(20, 'median') if largest else grouped.nsmallest(20, 'median')
    chosen = chosen.sort_values('median', ascending=False)
    y = np.arange(len(chosen))
    fig, ax = plt.subplots(figsize=(10, max(4, len(chosen) * 0.32)))
    ax.barh(y, chosen['median'].to_numpy(), color=TRAIN_COLOR)
    ax.axvline(MAX_FRAMES, color='black', linestyle='--')
    ax.set_yticks(y)
    ax.set_yticklabels([f'{name} (n={int(count)})' for name, count in zip(chosen.index, chosen['count'])])
    ax.invert_yaxis()
    ax.set_xlabel('Median số frame')
    ax.set_title(title)
    save_show(fig, filename)

plot_gloss_length(False, 'frames_shortest_glosses.png', '20 nhãn train có clip ngắn nhất')
plot_gloss_length(True, 'frames_longest_glosses.png', '20 nhãn train có clip dài nhất')


## 4. Điểm (0, 0, 0)

Điểm thiếu là đúng cả ba thành phần bằng 0, đúng cách file raw được ghi khi MediaPipe không ra landmark. Tỷ lệ theo khớp là số frame mà khớp đó vắng, chia cho số frame. Tỷ lệ hiện diện của một nhóm là số frame mà nhóm đó không bị trống hoàn toàn: thân trống cả 34 điểm, tay trống cả 21 điểm.


In [ ]:
for split in SPLITS:
    running = by_split[split]
    if running.frames <= 0:
        continue
    logger.info(
        '%s zero body=%.3f left=%.3f right=%.3f all=%.3f presence body=%.3f left=%.3f right=%.3f',
        split,
        zero_rate(running, BODY_INDEX),
        zero_rate(running, LEFT_INDEX),
        zero_rate(running, RIGHT_INDEX),
        zero_rate(running, np.arange(N_JOINTS)),
        running.body_present / running.frames,
        running.left_present / running.frames,
        running.right_present / running.frames,
    )

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(GROUP_NAMES))
width = 0.36
for offset, split, color in ((-width / 2, 'train', TRAIN_COLOR), (width / 2, 'test', TEST_COLOR)):
    running = by_split[split]
    if running.frames <= 0:
        continue
    rates = [zero_rate(running, GROUP_INDICES[group_id]) for group_id in range(3)]
    ax.bar(x + offset, rates, width=width, label=split, color=color)
ax.set_xticks(x)
ax.set_xticklabels(GROUP_NAMES)
ax.set_ylim(0, 1)
ax.set_ylabel('Tỷ lệ điểm (0, 0, 0)')
ax.set_title('Điểm thiếu theo nhóm khớp')
ax.legend()
save_show(fig, 'zeros_by_group.png')

train_running = by_split['train']
rate_source = train_running if train_running.frames > 0 else overall
rate_name = 'train' if train_running.frames > 0 else 'all'
joint_zero = rate_source.zero_counts / max(rate_source.frames, 1)
fig, ax = plt.subplots(figsize=(10, 18))
y = np.arange(N_JOINTS)
ax.barh(y, joint_zero, color=LANDMARK_COLORS)
if by_split['test'].frames > 0:
    test_zero = by_split['test'].zero_counts / by_split['test'].frames
    ax.scatter(test_zero, y, s=12, color='black', zorder=3, label='test')
    ax.legend()
ax.axvline(float(np.mean(joint_zero)), color='gray', linestyle='--')
ax.set_yticks(y)
ax.set_yticklabels(PRETTY_LANDMARKS, fontsize=7)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel('Tỷ lệ frame bị (0, 0, 0)')
ax.set_title(f'Tỷ lệ thiếu từng khớp ({rate_name}, chấm đen là test)')
save_show(fig, 'zeros_by_landmark.png')

worst_joints = np.argsort(joint_zero)[::-1][:10]
for joint in worst_joints:
    logger.info('Highest missing joint %s rate=%.3f', PRETTY_LANDMARKS[int(joint)], float(joint_zero[int(joint)]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split, color in (('train', TRAIN_COLOR), ('test', TEST_COLOR)):
    part = files_df[files_df['split'] == split]
    if part.empty:
        continue
    axes[0].hist(part['zero_ratio'], bins=np.linspace(0, 1, 21), alpha=0.55, label=split, color=color)
    axes[1].hist(part['right_presence'], bins=np.linspace(0, 1, 21), alpha=0.55, label=split, color=color)
axes[0].set_xlabel('Tỷ lệ điểm (0, 0, 0) trong clip')
axes[0].set_ylabel('Số clip')
axes[0].set_title('Clip bị thiếu bao nhiêu điểm')
axes[0].legend()
axes[1].set_xlabel('Tỷ lệ frame có tay phải')
axes[1].set_ylabel('Số clip')
axes[1].set_title('Tay phải có mặt trong bao nhiêu frame')
axes[1].legend()
save_show(fig, 'zeros_clip_hist.png')

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, column, title in zip(
    axes,
    ('body_presence', 'left_presence', 'right_presence'),
    ('Thân', 'Tay trái', 'Tay phải'),
):
    data = []
    labels = []
    for split in SPLITS:
        part = files_df.loc[files_df['split'] == split, column].to_numpy()
        if part.size == 0:
            continue
        data.append(part)
        labels.append(split)
    if not data:
        continue
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_title(title)
axes[0].set_ylabel('Tỷ lệ frame nhóm không trống')
save_show(fig, 'zeros_presence_box.png')

train_files = files_df[files_df['split'] == 'train']
if not train_files.empty:
    gloss_hand = train_files.groupby('gloss').agg(
        frames=('n_frames', 'sum'),
        left=('left_present_frames', 'sum'),
        right=('right_present_frames', 'sum'),
        clips=('n_frames', 'size'),
    )
    usable = gloss_hand[gloss_hand['clips'] >= 3]
    if usable.empty:
        usable = gloss_hand
    usable = usable.copy()
    usable['left_rate'] = usable['left'] / usable['frames']
    usable['right_rate'] = usable['right'] / usable['frames']
    worst = usable.nsmallest(25, 'right_rate').sort_values('right_rate', ascending=True)
    y = np.arange(len(worst))
    fig, ax = plt.subplots(figsize=(10, max(4, len(worst) * 0.32)))
    ax.barh(y - 0.2, worst['left_rate'], 0.4, label='tay trái', color=LEFT_COLOR)
    ax.barh(y + 0.2, worst['right_rate'], 0.4, label='tay phải', color=RIGHT_COLOR)
    ax.set_yticks(y)
    ax.set_yticklabels([f'{name} (n={int(count)})' for name, count in zip(worst.index, worst['clips'])])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel('Tỷ lệ frame có tay')
    ax.set_title('25 nhãn train tay phải vắng nhiều nhất')
    ax.legend()
    save_show(fig, 'zeros_worst_glosses.png')
    for gloss, row in worst.head(8).iterrows():
        logger.info(
            'Weak right hand %s clips=%d right=%.3f left=%.3f',
            gloss,
            int(row['clips']),
            float(row['right_rate']),
            float(row['left_rate']),
        )

fig, ax = plt.subplots(figsize=(7, 5))
for split, color in (('train', TRAIN_COLOR), ('test', TEST_COLOR)):
    part = files_df[files_df['split'] == split]
    if part.empty:
        continue
    ax.scatter(
        part['n_frames'],
        part['right_presence'],
        s=10,
        alpha=0.25,
        color=color,
        label=split,
        rasterized=True,
    )
ax.set_xlabel('Số frame')
ax.set_ylabel('Tỷ lệ frame có tay phải')
ax.set_ylim(-0.05, 1.05)
ax.set_title('Độ dài clip và mức bắt được tay phải')
ax.legend()
save_show(fig, 'zeros_frames_vs_hand.png')


## 5. Bộ x, y, z để chuẩn hoá

Chỉ tính trên điểm khác `(0, 0, 0)`. Nhét điểm thiếu vào mean sẽ kéo tâm về 0.

Bộ dùng khi train lấy từ **train**. `global` là một mean/std cho mọi khớp. `groups` tách thân, tay trái, tay phải vì `z` của tay (tương đối cổ tay) không cùng thang với `z` của thân (tương đối hông).

Z-score: `(v - mean) / std`. Sau đó gán lại `(0, 0, 0)` cho điểm thiếu. Phân vị p01/p99 dùng khi đuôi nặng hơn phân phối chuẩn: khoảng p01 đến p99 của chuẩn xấp xỉ `4.65 * std`; rộng hơn `6 * std` thì nên xét `(v - p01) / (p99 - p01)`.


In [ ]:
def summarize_axis(running: RunningStats, group_id, axis: int):
    if group_id is None:
        hist = running.hist[:, axis, :].sum(axis=0)
        under = int(running.underflow[:, axis].sum())
        over = int(running.overflow[:, axis].sum())
        indices = np.arange(N_JOINTS)
        group_name = 'all'
    else:
        hist = running.hist[group_id, axis]
        under = int(running.underflow[group_id, axis])
        over = int(running.overflow[group_id, axis])
        indices = GROUP_INDICES[group_id]
        group_name = GROUP_NAMES[group_id]
    count = int(running.valid_counts[indices].sum())
    hist_total = int(hist.sum()) + under + over
    if count != hist_total:
        logger.warning(
            'Count mismatch group=%s axis=%s points=%d hist=%d',
            group_name,
            AXIS_NAMES[axis],
            count,
            hist_total,
        )
    if count <= 0:
        return None
    total_sum = float(running.coord_sum[indices, axis].sum())
    total_sumsq = float(running.coord_sumsq[indices, axis].sum())
    mean = total_sum / count
    variance = max(total_sumsq / count - mean * mean, 0.0)
    std = math.sqrt(variance)
    vmin = float(np.min(running.coord_min[indices, axis]))
    vmax = float(np.max(running.coord_max[indices, axis]))
    if not math.isfinite(vmin):
        vmin = float('nan')
    if not math.isfinite(vmax):
        vmax = float('nan')
    outside = (under + over) / count
    if outside > 0.001:
        logger.warning(
            'Histogram clipped group=%s axis=%s fraction=%.4f min=%s max=%s',
            group_name,
            AXIS_NAMES[axis],
            outside,
            vmin,
            vmax,
        )
    return {
        'count': count,
        'mean': mean,
        'std': std,
        'min': vmin,
        'max': vmax,
        'p01': hist_percentile(hist, EDGES, under, over, 0.01),
        'p05': hist_percentile(hist, EDGES, under, over, 0.05),
        'p25': hist_percentile(hist, EDGES, under, over, 0.25),
        'p50': hist_percentile(hist, EDGES, under, over, 0.50),
        'p75': hist_percentile(hist, EDGES, under, over, 0.75),
        'p95': hist_percentile(hist, EDGES, under, over, 0.95),
        'p99': hist_percentile(hist, EDGES, under, over, 0.99),
        'fraction_outside_hist': outside,
        'fraction_below_0': hist_fraction_below(hist, EDGES, under, over, 0.0),
        'fraction_above_1': hist_fraction_above(hist, EDGES, under, over, 1.0),
        'hist': hist,
        'under': under,
        'over': over,
    }

def collect_summaries(running: RunningStats) -> dict:
    found = {'all': [summarize_axis(running, None, axis) for axis in range(3)]}
    for group_id, name in enumerate(GROUP_NAMES):
        found[name] = [summarize_axis(running, group_id, axis) for axis in range(3)]
    return found

def summary_frame(summary: dict, split_name: str) -> pd.DataFrame:
    rows = []
    for group_name in ('all',) + GROUP_NAMES:
        for axis, record in enumerate(summary[group_name]):
            if record is None:
                continue
            rows.append({
                'split': split_name,
                'group': group_name,
                'axis': AXIS_NAMES[axis],
                'count': record['count'],
                'mean': record['mean'],
                'std': record['std'],
                'min': record['min'],
                'p01': record['p01'],
                'p50': record['p50'],
                'p99': record['p99'],
                'max': record['max'],
                'outside_hist': record['fraction_outside_hist'],
                'below_0': record['fraction_below_0'],
                'above_1': record['fraction_above_1'],
            })
    return pd.DataFrame(rows)

if by_split['train'].files:
    fit = by_split['train']
    fit_name = 'train'
else:
    fit = overall
    fit_name = 'all'
    logger.warning('No train files. Normalization constants use every split.')

train_summary = collect_summaries(fit)
test_summary = collect_summaries(by_split['test']) if by_split['test'].files else None
display(summary_frame(train_summary, fit_name).round(4))
if test_summary is not None:
    display(summary_frame(test_summary, 'test').round(4))

shift_rows = []
if test_summary is not None:
    for group_name in ('all',) + GROUP_NAMES:
        for axis in range(3):
            train_axis = train_summary[group_name][axis]
            test_axis = test_summary[group_name][axis]
            if train_axis is None or test_axis is None or train_axis['std'] < 1e-8:
                continue
            delta = (test_axis['mean'] - train_axis['mean']) / train_axis['std']
            shift_rows.append({
                'group': group_name,
                'axis': AXIS_NAMES[axis],
                'train_mean': train_axis['mean'],
                'test_mean': test_axis['mean'],
                'train_std': train_axis['std'],
                'mean_shift_in_train_std': delta,
            })
shift_df = pd.DataFrame(shift_rows)
if not shift_df.empty:
    display(shift_df.round(4))

def pack_group(summary: dict, name: str, joints: np.ndarray) -> dict:
    records = summary[name]
    def column(key: str):
        values = []
        for record in records:
            values.append(None if record is None else record[key])
        return values
    return {
        'joints': joints.tolist(),
        'mean': column('mean'),
        'std': column('std'),
        'p01': column('p01'),
        'p50': column('p50'),
        'p99': column('p99'),
    }

joint_count = fit.valid_counts.astype(np.float64)
joint_mean = np.full((N_JOINTS, 3), np.nan, dtype=np.float64)
joint_std = np.full((N_JOINTS, 3), np.nan, dtype=np.float64)
present = joint_count > 0
joint_mean[present] = fit.coord_sum[present] / joint_count[present, None]
second = np.full((N_JOINTS, 3), np.nan, dtype=np.float64)
second[present] = fit.coord_sumsq[present] / joint_count[present, None]
variance = np.maximum(second - np.square(joint_mean), 0.0)
joint_std[present] = np.sqrt(variance[present])

recommended_groups = {
    name: pack_group(train_summary, name, GROUP_INDICES[group_id])
    for group_id, name in enumerate(GROUP_NAMES)
}
global_block = {
    'mean': [None if train_summary['all'][axis] is None else train_summary['all'][axis]['mean'] for axis in range(3)],
    'std': [None if train_summary['all'][axis] is None else train_summary['all'][axis]['std'] for axis in range(3)],
    'p01': [None if train_summary['all'][axis] is None else train_summary['all'][axis]['p01'] for axis in range(3)],
    'p50': [None if train_summary['all'][axis] is None else train_summary['all'][axis]['p50'] for axis in range(3)],
    'p99': [None if train_summary['all'][axis] is None else train_summary['all'][axis]['p99'] for axis in range(3)],
}

heavy_axes = []
for group_name in ('all',) + GROUP_NAMES:
    for axis, record in enumerate(train_summary[group_name]):
        if record is None or not math.isfinite(record['std']) or record['std'] < 1e-8:
            continue
        width = record['p99'] - record['p01']
        if math.isfinite(width) and width / record['std'] > 6:
            heavy_axes.append(f'{group_name}.{AXIS_NAMES[axis]}')

NORMALIZATION = {
    'fit_split': fit_name,
    'exclude': 'exact (0, 0, 0) points stay missing and are excluded from these stats',
    'formula_zscore': '(value - mean) / std, then write (0, 0, 0) back onto missing points',
    'formula_robust': '(value - p01) / (p99 - p01), then write (0, 0, 0) back onto missing points',
    'histogram_edges': [-4.0, 4.0],
    'histogram_step': 0.01,
    'global': global_block,
    'recommended': {
        'space': 'per_group',
        'method': 'zscore',
        'groups': recommended_groups,
    },
    'per_joint': {
        'names': LANDMARKS,
        'count': joint_count.astype(np.int64),
        'mean': joint_mean,
        'std': joint_std,
    },
    'heavy_tail_axes': heavy_axes,
    'summary': {
        'train_files': int(by_split['train'].files),
        'test_files': int(by_split['test'].files),
        'glosses': int(label_counts.shape[0]),
        'train_frames': int(by_split['train'].frames),
        'test_frames': int(by_split['test'].frames),
        'train_zero_rate': zero_rate(by_split['train'], np.arange(N_JOINTS)),
        'test_zero_rate': zero_rate(by_split['test'], np.arange(N_JOINTS)),
    },
}
json_path = OUTPUT_DIR / 'normalization_stats.json'
json_path.write_text(json.dumps(json_ready(NORMALIZATION), ensure_ascii=False, indent=2), encoding='utf-8')
logger.info('Wrote %s', json_path)
for name in GROUP_NAMES:
    block = recommended_groups[name]
    logger.info('%s mean=%s std=%s', name, block['mean'], block['std'])
logger.info('global mean=%s std=%s', global_block['mean'], global_block['std'])
if heavy_axes:
    logger.info('Heavy tails (p99-p01 > 6 std): %s', ', '.join(heavy_axes))

def finite_pair(record):
    if record is None:
        return None
    low = record['p01']
    high = record['p99']
    if not math.isfinite(low) or not math.isfinite(high) or high <= low:
        return None
    return low, high

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, ax in enumerate(axes):
    lows = []
    highs = []
    for group_id, name in enumerate(GROUP_NAMES):
        record = train_summary[name][axis]
        if record is None:
            continue
        density = density_from_hist(record['hist'], EDGES)
        ax.stairs(density, EDGES, fill=True, alpha=0.35, color=GROUP_COLORS[group_id], label=name)
        pair = finite_pair(record)
        if pair is not None:
            lows.append(pair[0])
            highs.append(pair[1])
    if lows and highs:
        low = min(lows)
        high = max(highs)
        pad = 0.08 * (high - low)
        ax.set_xlim(low - pad, high + pad)
    ax.set_title(AXIS_NAMES[axis])
    ax.set_xlabel(AXIS_NAMES[axis])
    ax.set_ylabel('Mật độ')
    ax.legend()
fig.suptitle(f'Mật độ {fit_name}, bỏ điểm (0, 0, 0)')
save_show(fig, 'xyz_group_density.png')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, ax in enumerate(axes):
    lows = []
    highs = []
    for split, running, color in (('train', by_split['train'], TRAIN_COLOR), ('test', by_split['test'], TEST_COLOR)):
        if running.files <= 0:
            continue
        record = summarize_axis(running, None, axis)
        if record is None:
            continue
        density = density_from_hist(record['hist'], EDGES)
        ax.stairs(density, EDGES, fill=True, alpha=0.35, color=color, label=split)
        pair = finite_pair(record)
        if pair is not None:
            lows.append(pair[0])
            highs.append(pair[1])
        if split == 'train' and math.isfinite(record['mean']):
            ax.axvline(record['mean'], color=TRAIN_COLOR, linestyle='--')
            if math.isfinite(record['std']) and record['std'] > 0:
                ax.axvline(record['mean'] - record['std'], color=TRAIN_COLOR, linestyle=':')
                ax.axvline(record['mean'] + record['std'], color=TRAIN_COLOR, linestyle=':')
    if lows and highs:
        low = min(lows)
        high = max(highs)
        pad = 0.08 * (high - low)
        ax.set_xlim(low - pad, high + pad)
    ax.set_title(AXIS_NAMES[axis])
    ax.set_xlabel(AXIS_NAMES[axis])
    ax.legend()
fig.suptitle('Train và test, mọi khớp. Nét đứt là mean train, nét chấm là mean ± std')
save_show(fig, 'xyz_split_density.png')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for axis, ax in enumerate(axes):
    for group_id, name in enumerate(GROUP_NAMES):
        record = train_summary[name][axis]
        if record is None:
            continue
        ax.hlines(group_id, record['p01'], record['p99'], color=GROUP_COLORS[group_id], linewidth=4)
        ax.plot(record['p50'], group_id, 'o', color='black')
        ax.plot(record['mean'], group_id, 'x', color='black')
    ax.set_yticks(range(len(GROUP_NAMES)))
    ax.set_yticklabels(GROUP_NAMES)
    ax.set_title(AXIS_NAMES[axis])
    ax.set_xlabel('Giá trị')
fig.suptitle(f'{fit_name}: đoạn p01-p99, chấm là median, dấu x là mean')
save_show(fig, 'xyz_ranges.png')

fig, axes = plt.subplots(1, 3, figsize=(14, 18), sharey=True)
y = np.arange(N_JOINTS)
for axis, ax in enumerate(axes):
    ax.barh(y, np.nan_to_num(joint_std[:, axis], nan=0.0), color=LANDMARK_COLORS)
    ax.set_xlabel(f'std {AXIS_NAMES[axis]}')
    ax.set_title(AXIS_NAMES[axis])
axes[0].set_yticks(y)
axes[0].set_yticklabels(PRETTY_LANDMARKS, fontsize=7)
axes[0].invert_yaxis()
fig.suptitle(f'Std từng khớp trên {fit_name}, bỏ điểm (0, 0, 0)')
save_show(fig, 'xyz_joint_std.png')

from matplotlib.lines import Line2D

presence = joint_count / max(fit.frames, 1)
keep = present & (presence >= 0.05) & np.isfinite(joint_mean).all(axis=1)
fig, ax = plt.subplots(figsize=(7, 8))
for start, end in BODY_EDGES:
    if keep[start] and keep[end]:
        ax.plot(joint_mean[[start, end], 0], joint_mean[[start, end], 1], color=BODY_COLOR, linewidth=1.5, zorder=1)
for side, color in ((0, LEFT_COLOR), (1, RIGHT_COLOR)):
    for chain in FINGER_CHAINS:
        joints = [hand_joint(local, side) for local in chain]
        for start, end in zip(joints, joints[1:]):
            if keep[start] and keep[end]:
                ax.plot(joint_mean[[start, end], 0], joint_mean[[start, end], 1], color=color, linewidth=1.2, zorder=1)
kept_index = np.flatnonzero(keep)
if kept_index.size:
    z_values = joint_mean[kept_index, 2]
    z_low = float(np.nanmin(z_values))
    z_high = float(np.nanmax(z_values))
    if not math.isfinite(z_low) or not math.isfinite(z_high) or z_high <= z_low:
        z_low, z_high = 0.0, 1.0
    norm = plt.Normalize(z_low, z_high)
    cmap = plt.get_cmap('coolwarm')
    for group_id, marker in ((0, 'o'), (1, 's'), (2, '^')):
        index = GROUP_INDICES[group_id]
        selected = index[keep[index]]
        if selected.size == 0:
            continue
        scatter = ax.scatter(
            joint_mean[selected, 0],
            joint_mean[selected, 1],
            c=joint_mean[selected, 2],
            cmap=cmap,
            norm=norm,
            marker=marker,
            s=36,
            zorder=2,
            edgecolors='black',
            linewidths=0.3,
        )
    colorbar = fig.colorbar(scatter, ax=ax)
    colorbar.set_label('z trung bình')
    xs = joint_mean[kept_index, 0]
    ys = joint_mean[kept_index, 1]
    ax.set_xlim(min(float(xs.min()), 0.0) - 0.05, max(float(xs.max()), 1.0) + 0.05)
    ax.set_ylim(max(float(ys.max()), 1.0) + 0.05, min(float(ys.min()), 0.0) - 0.05)
for joint in (0, 33, 34, 35):
    if keep[joint]:
        ax.annotate(
            PRETTY_LANDMARKS[joint],
            (joint_mean[joint, 0], joint_mean[joint, 1]),
            fontsize=8,
            xytext=(4, 4),
            textcoords='offset points',
        )
handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='gray', markeredgecolor='black', label='body'),
    Line2D([0], [0], marker='s', color='none', markerfacecolor='gray', markeredgecolor='black', label='left hand'),
    Line2D([0], [0], marker='^', color='none', markerfacecolor='gray', markeredgecolor='black', label='right hand'),
]
ax.legend(handles=handles, loc='best')
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('x')
ax.set_ylabel('y (đảo, giống nhìn ảnh)')
ax.set_title(f'Vị trí trung bình trên {fit_name}. Chỉ vẽ khớp có mặt ở ít nhất 5% frame.')
save_show(fig, 'xyz_mean_pose.png')

def normalize_keypoints(array: np.ndarray, stats: dict | None = None, method: str | None = None) -> np.ndarray:
    # Per-group affine norm. Exact (0, 0, 0) points are written back so missing stays missing.
    if stats is None:
        stats = NORMALIZATION
    chosen = stats['recommended']['method'] if method is None else method
    points = reshape_keypoints(array).astype(np.float32, copy=True)
    missing = np.all(points == 0, axis=-1)
    for name, index in (('body', BODY_INDEX), ('left_hand', LEFT_INDEX), ('right_hand', RIGHT_INDEX)):
        block = stats['recommended']['groups'][name]
        if chosen == 'robust':
            center = np.asarray(block['p01'], dtype=np.float32)
            scale = np.asarray(block['p99'], dtype=np.float32) - center
        elif chosen == 'global_zscore':
            center = np.asarray(stats['global']['mean'], dtype=np.float32)
            scale = np.asarray(stats['global']['std'], dtype=np.float32)
        else:
            center = np.asarray(block['mean'], dtype=np.float32)
            scale = np.asarray(block['std'], dtype=np.float32)
        scale = np.where(np.abs(scale) < np.float32(1e-6), np.float32(1.0), scale)
        points[:, index, :] = (points[:, index, :] - center) / scale
    points[missing] = 0
    return points

train_paths = files_df.loc[files_df['split'] == 'train', 'path']
sample_path = train_paths.iloc[0] if not train_paths.empty else files_df.iloc[0]['path']
raw_sample = reshape_keypoints(np.load(sample_path, allow_pickle=False)).astype(np.float32)
normed_sample = normalize_keypoints(raw_sample, NORMALIZATION)
missing_sample = np.all(raw_sample == 0, axis=-1)
if not np.all(normed_sample[missing_sample] == 0):
    logger.error('Normalization wrote into missing points: %s', sample_path)
else:
    logger.info('Missing points stay (0, 0, 0) after normalization: %s', sample_path)
valid_sample = ~missing_sample
became_zero = int(np.count_nonzero(np.all(normed_sample == 0, axis=-1) & valid_sample))
logger.info('Valid points that landed on exact zero: %d', became_zero)
if np.any(valid_sample):
    logger.info(
        'Sample valid mean raw=%s normed=%s',
        raw_sample[valid_sample].mean(axis=0).round(4).tolist(),
        normed_sample[valid_sample].mean(axis=0).round(4).tolist(),
    )

def fmt_xyz(values) -> str:
    parts = []
    for value in values:
        parts.append('nan' if value is None or not math.isfinite(float(value)) else f'{float(value):.4f}')
    return '[' + ', '.join(parts) + ']'

std_matrix = []
mean_matrix = []
for name in GROUP_NAMES:
    std_matrix.append([train_summary[name][axis]['std'] if train_summary[name][axis] else np.nan for axis in range(3)])
    mean_matrix.append([train_summary[name][axis]['mean'] if train_summary[name][axis] else np.nan for axis in range(3)])
std_matrix = np.asarray(std_matrix, dtype=np.float64)
mean_matrix = np.asarray(mean_matrix, dtype=np.float64)
std_ratio = std_matrix.max(axis=0) / np.maximum(std_matrix.min(axis=0), 1e-8)
mean_span = mean_matrix.max(axis=0) - mean_matrix.min(axis=0)

lines = [
    '## Kết luận',
    '',
    f"- Train **{int(by_split['train'].files)}** clip, test **{int(by_split['test'].files)}** clip, **{int(label_counts.shape[0])}** nhãn.",
    f"- Bộ chuẩn hoá fit trên **{fit_name}**, bỏ điểm đúng `(0, 0, 0)`.",
    f"- File số: `{json_path}`.",
    '',
    '### Bộ chung, mọi khớp',
    '',
    f"- mean x, y, z = `{fmt_xyz(global_block['mean'])}`",
    f"- std x, y, z = `{fmt_xyz(global_block['std'])}`",
    f"- p01 x, y, z = `{fmt_xyz(global_block['p01'])}`",
    f"- p99 x, y, z = `{fmt_xyz(global_block['p99'])}`",
    '',
    '### Bộ theo nhóm (nên dùng cái này)',
    '',
]
for name in GROUP_NAMES:
    block = recommended_groups[name]
    lines.append(f"- `{name}` mean `{fmt_xyz(block['mean'])}` std `{fmt_xyz(block['std'])}`")
lines.append('')
lines.append('Công thức đang chọn: `(v - mean) / std` theo từng nhóm và từng trục, rồi gán lại `(0, 0, 0)` cho điểm thiếu.')
lines.append('Muốn một bộ duy nhất cho mọi khớp thì gọi `normalize_keypoints(array, method="global_zscore")`.')
lines.append('')
for axis, axis_name in enumerate(AXIS_NAMES):
    global_std = train_summary['all'][axis]['std'] if train_summary['all'][axis] else float('nan')
    shared = std_ratio[axis] <= 1.35 and mean_span[axis] <= 0.5 * max(global_std, 1e-8)
    if shared:
        lines.append(f'- Trục `{axis_name}`: thân và hai tay gần cùng thang, một mean/std chung chấp nhận được.')
    else:
        lines.append(
            f'- Trục `{axis_name}`: std lớn nhất / nhỏ nhất giữa 3 nhóm là {std_ratio[axis]:.2f}, lệch mean {mean_span[axis]:.3f}. Tách nhóm.',
        )
if heavy_axes:
    lines.append('- Đuôi nặng (p99 - p01 > 6 std): ' + ', '.join(f'`{name}`' for name in heavy_axes) + '. Có thể đổi method `robust`.')
if not shift_df.empty:
    flagged = shift_df[shift_df['mean_shift_in_train_std'].abs() > 0.25]
    if flagged.empty:
        lines.append('- Mean test lệch train dưới 0.25 std trên mọi nhóm và mọi trục.')
    else:
        bits = [f"{row.group}.{row.axis}={row.mean_shift_in_train_std:.2f}" for row in flagged.itertuples()]
        lines.append('- Mean test lệch train quá 0.25 std: ' + ', '.join(bits) + '.')
x_all = train_summary['all'][0]
y_all = train_summary['all'][1]
if x_all is not None and y_all is not None:
    lines.append(
        f"- Điểm hợp lệ có x < 0: {x_all['fraction_below_0']:.2%}, x > 1: {x_all['fraction_above_1']:.2%}. y < 0: {y_all['fraction_below_0']:.2%}, y > 1: {y_all['fraction_above_1']:.2%}.",
    )
over_long = files_df['n_frames'] > MAX_FRAMES
lines.append(
    f"- Clip dài hơn 80 frame: {int(over_long.sum())} / {len(files_df)} ({over_long.mean():.1%}). Phần dư sẽ bị cắt nếu giữ pipeline 80 frame.",
)
lines.append(
    f"- Tỷ lệ điểm (0, 0, 0) train {zero_rate(by_split['train'], np.arange(N_JOINTS)):.2%}, test {zero_rate(by_split['test'], np.arange(N_JOINTS)):.2%}.",
)
display(Markdown('\n'.join(lines)))
